## Embeddings

Hamare pichhle example mein, humne high-dimensional bag-of-words vector par kaam kiya tha, jiski length `vocab_size` thi, aur hum explicitly low-dimensional positional representation vector ko sparse one-hot representation mein convert kar rahe the. Yeh one-hot representation memory-efficient nahi hai, aur iske alawa, har word ko ek-doosre se independent maana jaata hai, yaani one-hot encoded vectors words ke beech kisi bhi semantic similarity ko express nahi karte.

Is unit mein, hum **News AG** dataset ko explore karna jaari rakhenge. Shuru karne ke liye, chaliye data load karte hain aur pichhle notebook se kuch definitions le lete hain.

In [ ]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

## Embedding kya hai?

**Embedding** ka idea yeh hai ki words ko low-dimensional dense vectors se represent kiya jaye, jo kisi tarah word ke meaning ko reflect karte hain. Meaningful word embeddings kaise banayein, iske baare mein hum baad mein baat karenge, lekin abhi ke liye, embeddings ko word vector ki dimensionality kam karne ke ek tareeke ke roop mein sochiye.

Toh, embedding layer ek word ko input ke roop mein legi aur specified `embedding_size` ka output vector produce karegi. Ek tarah se, yeh `Linear` layer jaisi hi hai, lekin one-hot encoded vector lene ke bajaye, yeh ek word number ko input ke roop mein le sakegi.

Apne network mein embedding layer ko pehli layer ke roop mein use karke, hum bag-of-words se **embedding bag** model par switch kar sakte hain, jahan hum pehle apne text ke har word ko uski corresponding embedding mein convert karte hain, aur phir un saari embeddings par koi aggregate function calculate karte hain, jaise `sum`, `average` ya `max`.

![Paanch sequence words ke liye embedding classifier dikhati image.](../../../../../lessons/5-NLP/14-Embeddings/images/embedding-classifier-example.png)

Hamara classifier neural network embedding layer se shuru hoga, phir aggregation layer, aur uske upar ek linear classifier:

In [ ]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Variable sequence sizes se deal karna

Is architecture ki wajah se, hamare network ke liye minibatches ko ek khaas tareeke se banana hoga. Pichhle unit mein, jab bag-of-words (BoW) use kar rahe the, toh minibatch mein saare BoW tensors ka size `vocab_size` ke barabar hota tha, chahe hamare text sequence ki actual length kuch bhi ho. Jab hum word embeddings par jaate hain, toh har text sample mein words ki sankhya alag-alag ho sakti hai, aur in samples ko minibatches mein combine karte waqt humein kuch padding apply karni hogi.

Yeh `collate_fn` function ko data source mein provide karne ki same technique use karke kiya ja sakta hai:

In [ ]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Embedding classifier ko train karna

Ab jab humne sahi dataloader define kar liya hai, toh hum pichhle unit mein define kiye gaye training function ka use karke model ko train kar sakte hain:

In [ ]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

> **Note**: Hum yahan time bachane ke liye sirf 25k records (ek full epoch se kam) ke liye train kar rahe hain, lekin aap training jaari rakh sakte hain, kai epochs ke liye training karne ke liye ek function likh sakte hain, aur higher accuracy paane ke liye learning rate parameter ke saath experiment kar sakte hain. Aapko lagbhag 90% accuracy tak pahunchne mein saksham hona chahiye.

### EmbeddingBag layer aur variable length sequence representation

Pichhli architecture mein, humein saare sequences ko ek hi length tak pad karna padta tha taaki unhe minibatch mein fit kiya ja sake. Yeh variable length sequences ko represent karne ka sabse efficient tareeka nahi hai - ek aur approach **offset** vector use karna hoga, jo ek bade vector mein stored saare sequences ke offsets ko rakhega.

![Offset sequence representation dikhati image](../../../../../lessons/5-NLP/14-Embeddings/images/offset-sequence-representation.png)

> **Note**: Upar di gayi picture mein, humne characters ka sequence dikhaya hai, lekin hamare example mein hum words ke sequences ke saath kaam kar rahe hain. Halaanki, offset vector ke saath sequences ko represent karne ka general principle wahi rehta hai.

Offset representation ke saath kaam karne ke liye, hum [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) layer use karte hain. Yeh `Embedding` jaisi hi hai, lekin yeh content vector aur offset vector ko input ke roop mein leti hai, aur ismein average layer bhi shaamil hoti hai, jo `mean`, `sum` ya `max` ho sakti hai.

Yeh ek modified network hai jo `EmbeddingBag` use karta hai:

In [ ]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Dataset ko training ke liye taiyaar karne ke liye, humein ek conversion function provide karna hoga jo offset vector taiyaar karega:

In [ ]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Dhyan dijiye ki pichhle saare examples ke vipreet, hamara network ab do parameters accept karta hai: data vector aur offset vector, jo alag-alag sizes ke hote hain. Isi tarah, hamara dataloader bhi humein 2 ke bajaye 3 values deta hai: text aur offset vector dono ko features ke roop mein diya jaata hai. Isliye, humein apne training function ko thoda adjust karna hoga taaki iska dhyan rakha ja sake:

In [ ]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

## Semantic Embeddings: Word2Vec

Pichhle example mein, model embedding layer ne words ko vector representation mein map karna seekha, lekin is representation mein zyada semantic meaning nahi tha. Achha hoga agar aisa vector representation seekha jaye, jisme similar words ya synonyms aise vectors se match karein jo kisi vector distance (jaise, Euclidean distance) ke hisaab se ek-doosre ke kareeb hon.

Iske liye, humein apne embedding model ko ek bade text collection par ek khaas tareeke se pre-train karna hoga. Semantic embeddings ko train karne ke shuruaati tareekon mein se ek ko [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) kehte hain. Yeh words ke distributed representations banane ke liye do main architectures par based hai:

 - **Continuous bag-of-words** (CBoW) — is architecture mein, hum model ko aas-paas ke context se ek word predict karne ke liye train karte hain. Diye gaye ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ mein, model ka goal $(W_{-2},W_{-1},W_1,W_2)$ se $W_0$ ko predict karna hai.
 - **Continuous skip-gram** CBoW ka ulta hai. Model context words ki aas-paas ki window use karke current word ko predict karta hai.

CBoW fast hai, jabki skip-gram slow hai, lekin yeh kam use hone wale words ko behtar represent karta hai.

![CBoW aur skip-gram algorithms ko words ko vectors mein convert karte hue dikhati image.](../../../../../lessons/5-NLP/14-Embeddings/images/example-algorithms-for-converting-words-to-vectors.png)

Google News dataset par pre-trained Word2Vec embedding ke saath experiment karne ke liye, hum **gensim** library use kar sakte hain. Neeche hum 'neural' ke sabse similar words dhoondhte hain.

> **Note:** Jab aap pehli baar word vectors banate hain, toh unhe download hone mein kuch time lag sakta hai!

In [ ]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [ ]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

Hum word se vector embedding bhi calculate kar sakte hain, jise classification model ko train karne mein use kiya ja sakta hai (clarity ke liye hum vector ke sirf pehle 20 components dikhate hain):

In [ ]:
w2v.word_vec('play')[:20]

In [ ]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

CBoW aur Skip-Grams dono "predictive" embeddings hain, kyunki yeh sirf local contexts ko dhyan mein rakhte hain. Word2Vec global context ka fayda nahi uthata.

**FastText**, Word2Vec par based hai aur har word aur uske andar paaye jaane wale character n-grams ke liye vector representation seekhta hai. In representations ki values ko har training step mein ek vector mein average kiya jaata hai. Halaanki yeh pre-training mein kaafi extra computation add karta hai, lekin yeh word embeddings ko sub-word information encode karne ke kaabil banata hai.

Ek aur method, **GloVe**, co-occurrence matrix ke idea ka use karta hai aur co-occurrence matrix ko zyada expressive aur non-linear word vectors mein factorize karne ke liye neural methods use karta hai.

Aap example ke saath khel sakte hain aur embeddings ko FastText aur GloVe mein badal sakte hain, kyunki gensim kai alag-alag word embedding models ko support karta hai.

## PyTorch mein pre-trained embeddings use karna

Hum upar wale example ko modify kar sakte hain taaki hamari embedding layer ke matrix ko pehle se taiyaar kiye gaye semantic embeddings, jaise Word2Vec, se pre-populate kiya ja sake. Humein yeh dhyan rakhna hoga ki pre-trained embedding aur hamare text corpus ki vocabularies shayad match nahi karengi, isliye hum un words ke liye weights ko random values se initialize karenge jo missing hain:

In [ ]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

In [ ]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

Hamare case mein, humein accuracy mein bahut zyada badhotri nahi dikhti, jo shayad alag-alag vocabulary ki wajah se hai.
Alag-alag vocabulary ki problem ko solve karne ke liye, hum neeche diye gaye solutions mein se ek use kar sakte hain:
* Hamari vocabulary par Word2Vec model ko dobara train karein
* Pre-trained Word2Vec model ki vocabulary ke saath apna dataset load karein. Dataset load karne ke liye use ki jaane wali vocabulary ko loading ke dauraan specify kiya ja sakta hai.

Doosra tareeka zyada aasaan lagta hai, khaas taur par kyunki PyTorch `torchtext` framework mein embeddings ke liye built-in support hai. Jaise, hum GloVe-based vocabulary ko is tarah instantiate kar sakte hain:

In [ ]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

Loaded vocabulary mein neeche diye gaye basic operations hote hain:
* `vocab.stoi` dictionary humein kisi word ko uske dictionary index mein convert karne deti hai
* `vocab.itos` iska ulta karta hai - number ko word mein convert karta hai
* `vocab.vectors` embedding vectors ka array hai, isliye kisi word `s` ki embedding paane ke liye humein `vocab.vectors[vocab.stoi[s]]` use karna hoga

Yahan embeddings ke saath chhed-chhad ka ek example hai, jo equation **kind-man+woman = queen** ko demonstrate karta hai (ise kaam karwane ke liye mujhe coefficient thoda adjust karna pada):

In [ ]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

In embeddings ka use karke classifier ko train karne ke liye, humein pehle apne dataset ko GloVe vocabulary use karke encode karna hoga:

In [ ]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Jaisa humne upar dekha, saari vector embeddings `vocab.vectors` matrix mein stored hoti hain. Isse ise embedding layer ke weights mein simple copying ke zariye load karna bahut aasaan ho jaata hai:

In [ ]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

Hamari accuracy mein significant badhotri na dikhne ke reasons mein se ek yeh hai ki hamare dataset ke kuch words pre-trained GloVe vocabulary mein nahi hain, aur isliye unhe ignore kar diya jaata hai. Is problem ko door karne ke liye, hum apne dataset par apni khud ki embeddings train kar sakte hain.

## Contextual Embeddings

Traditional pretrained embedding representations, jaise Word2Vec, ki ek main limitation word sense disambiguation ki problem hai. Jabki pretrained embeddings context mein words ka kuch meaning capture kar sakti hain, ek word ke har possible meaning ko ek hi embedding mein encode kiya jaata hai. Isse downstream models mein problems aa sakti hain, kyunki kai words, jaise 'play', ke alag-alag contexts mein alag-alag meanings ho sakte hain.

Jaise, 'play' word ka in do sentences mein kaafi alag meaning hai:
- Main theatre mein ek **play** dekhne gaya.
- John apne doston ke saath **play** karna chahta hai.

Upar diye gaye pretrained embeddings 'play' word ke in dono meanings ko ek hi embedding mein represent karte hain. Is limitation ko door karne ke liye, humein **language model** par based embeddings banani hongi, jo ek bade text corpus par train hota hai aur *jaanta hai* ki words ko alag-alag contexts mein kaise jodha ja sakta hai. Contextual embeddings par baat karna is tutorial ke scope se bahar hai, lekin hum agle unit mein language models par baat karte waqt isse wapas aayenge.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->